# DSAI 413 — A2: Notebook 1 — Data Exploration

**Goal:** Load a subset of MIMIC-CXR, explore image quality and report structure, and prepare the data for downstream tasks.

**What this notebook does:**
1. Mount Google Drive / install dependencies
2. Load the MIMIC-CXR metadata CSV
3. Visualize sample X-ray images
4. Analyze report structure (length, section presence)
5. Build the train/val/test split
6. Save the processed subset for use in other notebooks

In [ ]:
# ── 0. Install dependencies (run once in Colab) ───────────────────────────
# !pip install -r ../requirements.txt --quiet

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter

from src.config import (
    MIMIC_CSV_PATH, MIMIC_IMG_COL, MIMIC_TEXT_COL,
    DATASET_SUBSET_SIZE, SPLIT_RATIOS
)
from src.preprocessing import load_mimic_subset

print('Imports OK')

## 1. Load metadata CSV

In [ ]:
# Update MIMIC_CSV_PATH in src/config.py to point to your downloaded CSV
df = pd.read_csv(MIMIC_CSV_PATH)
print(f'Total rows: {len(df):,}')
print(f'Columns: {list(df.columns)}')
df.head(3)

## 2. Sample images

In [ ]:
from src.preprocessing import load_image

sample_df = df.dropna(subset=[MIMIC_IMG_COL, MIMIC_TEXT_COL]).sample(9, random_state=42)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for ax, (_, row) in zip(axes.flatten(), sample_df.iterrows()):
    try:
        img = load_image(row[MIMIC_IMG_COL])
        ax.imshow(img, cmap='gray')
        ax.set_title(str(row[MIMIC_IMG_COL])[-30:], fontsize=7)
    except Exception as e:
        ax.set_title(f'Error: {e}', fontsize=7)
    ax.axis('off')

plt.suptitle('Sample MIMIC-CXR Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Report structure analysis

In [ ]:
reports = df[MIMIC_TEXT_COL].dropna().astype(str)

# Report length distribution
lengths = reports.str.split().apply(len)
print(f'Report length — mean: {lengths.mean():.0f} words, median: {lengths.median():.0f}, max: {lengths.max()}')

# Section presence
has_findings   = reports.str.upper().str.contains('FINDINGS').sum()
has_impression = reports.str.upper().str.contains('IMPRESSION').sum()
print(f'Reports with FINDINGS section:   {has_findings:,} ({has_findings/len(reports)*100:.1f}%)')
print(f'Reports with IMPRESSION section: {has_impression:,} ({has_impression/len(reports)*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(lengths, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Report Length Distribution (words)')
axes[0].set_xlabel('Words')
axes[0].set_ylabel('Count')

section_counts = {'FINDINGS': has_findings, 'IMPRESSION': has_impression, 'Neither': len(reports) - has_findings}
axes[1].bar(section_counts.keys(), section_counts.values(), color=['#0d6efd', '#28a745', '#dc3545'])
axes[1].set_title('Section Presence in Reports')
plt.tight_layout()
plt.show()

## 4. Load subset and create train/val/test split

In [ ]:
from sklearn.model_selection import train_test_split

images, reports, img_paths = load_mimic_subset(
    MIMIC_CSV_PATH,
    img_col=MIMIC_IMG_COL,
    text_col=MIMIC_TEXT_COL,
    subset_size=DATASET_SUBSET_SIZE,
)

# Split
n = len(images)
indices = list(range(n))
train_idx, temp_idx = train_test_split(indices, test_size=1 - SPLIT_RATIOS['train'], random_state=42)
val_idx, test_idx   = train_test_split(temp_idx, test_size=0.5, random_state=42)

print(f'Split sizes — Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}')

# Save split indices for reuse
import json
os.makedirs('../data', exist_ok=True)
with open('../data/split_indices.json', 'w') as f:
    json.dump({'train': train_idx, 'val': val_idx, 'test': test_idx}, f)
print('Split indices saved to ../data/split_indices.json')

## 5. Keyword frequency in reports

In [ ]:
from src.qa_dataset_creation import CLINICAL_TERMS

term_counts = {}
for term in CLINICAL_TERMS:
    count = sum(term.lower() in r.lower() for r in reports)
    term_counts[term] = count

term_counts_sorted = dict(sorted(term_counts.items(), key=lambda x: -x[1]))

plt.figure(figsize=(12, 5))
plt.bar(term_counts_sorted.keys(), term_counts_sorted.values(), color='teal', edgecolor='white')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.title('Clinical Term Frequency in MIMIC-CXR Subset')
plt.ylabel('Count')
plt.tight_layout()
plt.show()